**Extracted Data, Findings, and Mappings for Healthcare Access Features (Nepal NDHS 2022 & Related Reports):**


#### 29. **distance_to_health_facility** (in km)
- **Statistics:** Median distance is ~2.1 km nationally, but huge variance.
    - **Urban:** median ~0.8 km, IQR: 0.5–2.5 km.
    - **Rural:** median ~4.5 km, IQR: 2–15 km. Remote/karnali provinces: up to 50 km.
- **Distribution:** Log-normal, strong right skew; mode ~1 km, mean ~3.2 km.  
- **Mapping:** Rural households much further than urban; distance increases with poverty, poorer provinces.


#### 30. **health_insurance** {Yes, No}
- **Statistics:** NDHS 2022, MoHP: ~30% of households have some form of government health insurance coverage (rapidly growing; target is universal in 5 years).
    - Higher coverage in urban, wealthier families, Bagmati, Gandaki.
    - Lower in marginalized, poorest, rural provinces.

#### 31. **previous_mental_health_consultation** {Yes, No}
- **Statistics:** NDHS 2022, BMJ Open: Only ~5% of Nepalis reporting depression/anxiety symptoms have ever consulted a mental health professional; among mothers, likely even lower (~3–5%).[5][6]
    - More likely if educated, urban, history of violence, chronic illness.
    - Major barriers: stigma, access, service availability.

### **Mapping and Relationships**

| Variable                          | Strongly Related to...                        | Observed Nepal Patterns                                     |
|------------------------------------|-----------------------------------------------|-------------------------------------------------------------|
| **distance_to_health_facility**    | residence, wealth, province                   | Rural, poor, hill/mountain provinces ↑ distance; urban ↓    |
| **health_insurance**               | wealth, residence, education, province        | Urban, higher wealth/edu, Bagmati/Gandaki ↑; poor/rural ↓   |
| **previous_mental_health_consultation** | education, urban residence, violence, sleep quality | ↑ among educated, urban, exposed to violence; very low in poor, rural |

#### **Summary Table (Feature Mapping for Nepal)**

| residence | wealth | province    | distance_to_facility | health_insurance | mental_health_consult |
|-----------|-------|-------------|----------------------|------------------|----------------------|
| Urban     | Rich  | Bagmati     | 0.5–2 km             | 40–50% Yes       | 7–10% Yes            |
| Rural     | Poor  | Karnali     | 5–50 km              | 15–25% Yes       | 2–4% Yes             |
| Urban     | Any   | Any         | 0.5–4 km             | 29% Yes (avg)    | 5% Yes               |
| Rural     | Any   | Any         | 2–50 km              | 22% Yes (avg)    | 3% Yes               |

**References:** NDHS 2022, MoHP, BMJOpen, PublicHealthConcernNepal, CollegeNP, JOGHR, WHO Nepal national reports.

----
---
------
------

In [2]:
import numpy as np
import pandas as pd

n_samples = 15000

# Simulate context columns: residence, wealth, province (can be merged later)
residences = ["Urban", "Rural"]
resid_probs = [0.65, 0.35]
wealth_cats = ['Poorest','Poorer','Middle','Richer','Richest']
wealth_p = [0.202,0.202,0.202,0.197,0.197]
provinces = ['Koshi', 'Madhesh', 'Bagmati', 'Gandaki', 'Lumbini', 'Karnali', 'Sudurpashchim']
prov_p = [0.16, 0.15, 0.18, 0.12, 0.15, 0.12, 0.12]

def dist_to_facility(residence, province, wealth):
    # All values in km – log-normal; rural, Karnali, Sudurpashchim much higher
    base = 1.1 if residence=="Urban" else 3.1
    if province == "Karnali":
        base *= 3.8
    elif province == "Sudurpashchim":
        base *= 2.2
    elif province == "Bagmati":
        base *= 0.6
    elif province == "Madhesh":
        base *= 0.85
    # Wealth effect (richer → urban proximity)
    if wealth in ("Richest","Richer"):
        base *= 0.7
    elif wealth in ("Poorest","Poorer"):
        base *= 1.2
    log_mu = np.log(base)
    log_sigma = 0.7
    dist = np.random.lognormal(log_mu, log_sigma)
    return np.clip(dist, 0.5, 50)

def health_insurance(residence, wealth):
    # NDHS: 30% covered, more likely in urban/wealthier
    p = 0.50 if residence=="Urban" and wealth in ("Richest","Richer") else \
        0.35 if residence=="Urban" else \
        0.25 if wealth in ("Richest","Richer") else 0.19
    return "Yes" if np.random.rand() < p else "No"

def mental_health_consult(residence, wealth, education=None):
    # Education can bump rates; urban/wealth ↑
    p = 0.10 if residence=="Urban" and wealth in ("Richest","Richer") else \
        0.07 if residence=="Urban" else \
        0.04 if wealth in ("Richest","Richer") else 0.025
    return "Yes" if np.random.rand() < p else "No"

rows = []
for _ in range(n_samples):
    residence = np.random.choice(residences, p=resid_probs)
    wealth = np.random.choice(wealth_cats, p=wealth_p)
    province = np.random.choice(provinces, p=prov_p)
    dist = dist_to_facility(residence, province, wealth)
    insurance = health_insurance(residence, wealth)
    consult = mental_health_consult(residence, wealth)
    rows.append([
        round(dist,2),
        insurance,
        consult,
        residence,
        wealth,
        province
    ])

columns = [
    "distance_to_health_facility",
    "health_insurance",
    "previous_mental_health_consultation",
    "residence",
    "wealth_index",
    "province"
]

df = pd.DataFrame(rows, columns=columns)

In [3]:
df.head()

,distance_to_health_facility,health_insurance,previous_mental_health_consultation,residence,wealth_index,province
0,8.43,No,No,Rural,Poorer,Sudurpashchim
1,2.32,No,Yes,Urban,Poorest,Lumbini
2,3.60,Yes,No,Urban,Richer,Karnali
3,1.25,No,No,Urban,Poorer,Madhesh
4,2.13,No,No,Urban,Poorer,Koshi


- **distance_to_health_facility:** Higher in rural (esp. Karnali, Sudurpashchim, poor), lowest in urban/Bagmati/rich.

- **health_insurance:** 30% overall, driven by urban/richness.

- **previous_mental_health_consultation:** Nationally very low; peaks in educated urban rich (max ~10%).